# Configuration

## Install necessary libraries from python

In [1]:
import os
import pickle
import subprocess
import shutil

In [2]:
# %%capture
!pip install transformers datasets torch
!pip install git+https://github.com/huggingface/accelerate
# !pip install dgl==2.0.0 -f https://data.dgl.ai/wheels/cu121/repo.html
!sudo apt-get -q install graphviz graphviz-dev
!pip install -q pygraphviz
!pip install -q slither-analyzer==0.8.0
!pip install dgl==1.1.2
!pip install py-solc==3.2.0
!pip install networkx==2.5.1
!solc-select install 0.4.25
!solc-select use 0.4.25
!which solc && solc --version

  Cloning https://github.com/huggingface/accelerate to /tmp/pip-req-build-r_ysnxrv
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/accelerate /tmp/pip-req-build-r_ysnxrv
  Resolved https://github.com/huggingface/accelerate to commit 4677b8089f6f1cf0eb39eb9b3a8f1188e9f1afe8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for accelerate: filename=accelerate-1.4.0.dev0-py3-none-any.whl size=343080 sha256=8b6dac3c3131930023da20df45a5dbc4cf6e5cbdbd0b81378989f67ec90c5e16
  Stored in directory: /tmp/pip-ephem-wheel-cache-2kktauaj/wheels/f6/c7/9d/1b8a5ca8353d9307733bc719107acb67acdc95063bba749f26
Successfully built accelerate
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.2.1
    Uninstalling accelerate-1.2.1:
      Successfully uninstalled accelerate-1.2.1
Reading package lists...
Building dependency tree...
Reading stat

In [3]:
# %%capture
file_path = '/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/sc_versions.pkl'
with open(file_path, 'rb') as f:
    sc_versions = pickle.load(f)

destination_path = '/content/ge-sc/artifacts'
for sc_version in sc_versions:

    print(sc_version)
    try:
        subprocess.run(['solc-select', 'install', sc_version])
        solc_compiler = os.path.expanduser(f'~/.solc-select/artifacts/solc-{sc_version}')
        shutil.copytree(solc_compiler, destination_path, dirs_exist_ok=True)
    except Exception as e:
        print(sc_version)
        print(e)

0.4.20
0.4.99
0.4.99
[Errno 2] No such file or directory: '/root/.solc-select/artifacts/solc-0.4.99'
0.4.9
0.4.22
0.4.19
0.4.2
0.4.11
0.4.10
0.4.23
0.5.5
0.5.2
0.5.7
0.5.0
0.4.8
0.4.7
0.5.8
0.4.4
0.4.15
0.4.24
0.4.12
0.4.16
0.4.13
0.4.26
0.5.9
0.5.4
0.5.6
0.4.0
0.5.3
0.4.21
0.5.1
0.4.17
0.8.0
0.4.18
0.4.25
0.4.14
0.4.6


## Import Python libraries

In [4]:
from concurrent.futures import ThreadPoolExecutor
from random import sample
import json
import multiprocessing
import traceback
import pandas as pd
import os
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import dgl
import os
import shutil
import random
import os

import re
import logging
# import pygraphviz as pgv
import networkx as nx

from copy import deepcopy
from os.path import join
from scipy.integrate._ivp.radau import C
from slither.slither import Slither
from collections import defaultdict
from networkx.algorithms import cluster
from slither.core.cfg.node import Node, NodeType
from tqdm import tqdm

from slither.printers.call import call_graph
# from slither.printers.summary.constructor_calls import _get_source_code
from slither.printers.abstract_printer import AbstractPrinter
from slither.core.declarations.solidity_variables import SolidityFunction
from slither.core.declarations.function import Function
from slither.core.variables.variable import Variable

from pathlib import Path
import glob
from multiprocessing import Pool as ThreadPool
from functools import partial
import torch.nn as nn
import torch
from torch import Tensor
import torch.nn.functional as F
from dgl.nn import GraphConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
import pickle
import random
import numpy as np
from tqdm import tqdm
import os
import json

from os.path import join
from shutil import copy
from re import L
from typing import Pattern
from tqdm import tqdm
import re
# from slither.core.cfg.node import NodeType
# from solc import install_solc

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


# Solidity code to Graph

## Generate call-graph

In [5]:
logger = logging.getLogger("Slither-simil")

# get sol version
def get_solc_version(source):
    pattern =  re.compile(r'\d.\d.\d+')
    with open(source, 'r') as f:
        line = f.readline()
        while line:
            if 'pragma solidity' in line:
                if len(pattern.findall(line)) > 0:
                    return pattern.findall(line)[0]
                else:
                    return '0.4.25'
            line = f.readline()
    return '0.4.25'

# Contract function node
def _function_node(contract, function, filename_input):
    node_function_source_code_start = function.source_mapping['start']
    node_function_source_code_length = function.source_mapping['length']
    node_info = {
        'node_id': f"{filename_input}_{contract.id}_{contract.name}_{function.full_name}",
        'label': f"{filename_input}_{contract.name}_{function.full_name}",
        'function_fullname': function.full_name,
        'contract_name': contract.name,
        'source_file': filename_input,
        'node_source_code_start': node_function_source_code_start,
        'node_source_code_length': node_function_source_code_length,
        'visibility': function.visibility
    }
    return node_info

# Solidity function node
def _solidity_function_node(solidity_function):
    node_info = {
        'node_id': f"[Solidity]_{solidity_function.full_name}",
        'label': f"[Solidity]_{solidity_function.full_name}",
        'function_fullname': solidity_function.full_name,
        'contract_name': None,
        'source_file': None,
        'node_source_code_start': None,
        'node_source_code_length': None,
        'visibility': 'public'
    }
    return node_info

# return node info from a node tupple
def _get_node_info(tuple_node):
    if tuple_node[0][0] == 'node_id':
        node_id = tuple_node[0][1]
    if tuple_node[1][0] == 'label':
        node_label = tuple_node[1][1]
    if tuple_node[2][0] == 'function_fullname':
        function_fullname = tuple_node[2][1]
    if tuple_node[3][0] == 'contract_name':
        contract_name = tuple_node[3][1]
    if tuple_node[4][0] == 'source_file':
        source_file = tuple_node[4][1]
    if tuple_node[5][0] == 'node_source_code_start':
        node_function_source_code_start = tuple_node[5][1]
    if tuple_node[6][0] == 'node_source_code_length':
        node_function_source_code_length = tuple_node[6][1]
    if tuple_node[7][0] == 'visibility':
        visibility = tuple_node[7][1]

    if 'fallback' in node_id:
        node_type = 'fallback_function'
    elif '[Solidity]' in node_id:
        node_type = 'fallback_function'
    else:
        node_type = 'contract_function'

    return node_id, node_label, node_type, function_fullname, contract_name, source_file, node_function_source_code_start, node_function_source_code_length, visibility

# return edge info from a contract call tuple
def _add_edge_info_to_nxgraph(contract_call, nx_graph):
    source = contract_call[0]
    source_node_id, source_label, source_type, source_function_fullname, source_contract_name, \
    source_source_file, source_node_function_source_code_start, source_node_function_source_code_length, source_visibility = _get_node_info(source)

    if source_node_id not in nx_graph.nodes():
        nx_graph.add_node(source_node_id, label=source_label, node_type=source_type,
                          node_source_code_start=source_node_function_source_code_start, node_source_code_length=source_node_function_source_code_length,
                          function_fullname=source_function_fullname,
                          function_vis=source_visibility, contract_name=source_contract_name,
                          source_file=source_source_file)

    target = contract_call[1]
    target_node_id, target_label, target_type, target_function_fullname, target_contract_name, \
    target_source_file, target_node_function_source_code_start, target_node_function_source_code_length,  target_visibility = _get_node_info(target)

    if target_node_id not in nx_graph.nodes():
        nx_graph.add_node(target_node_id, label=target_label, node_type=target_type,
                          node_source_code_start=target_node_function_source_code_start, node_source_code_length=target_node_function_source_code_length,
                          function_fullname=target_function_fullname,
                          function_vis=target_visibility, contract_name=target_contract_name,
                          source_file=target_source_file)

    edge_type = contract_call[2]
    edge_label = contract_call[3]

    nx_graph.add_edge(source_node_id, target_node_id, label=edge_label, edge_type=edge_type)

def _process_internal_call(
    contract,
    function,
    internal_call,
    contract_calls,
    solidity_functions,
    solidity_calls,
    filename_input
):
    if isinstance(internal_call, (Function)):
        contract_calls[contract].add(
            (
                tuple(_function_node(contract, function, filename_input).items()),
                tuple(_function_node(contract, internal_call, filename_input).items()),
                'internal_call',
                'internal_call'
            )
        )

    elif isinstance(internal_call, (SolidityFunction)):
        solidity_functions.add(tuple(_solidity_function_node(internal_call).items()))
        solidity_calls.add(
            (
                tuple(_function_node(contract, function, filename_input).items()),
                tuple(_solidity_function_node(internal_call).items()),
                'solidity_call',
                'solidity_call'
            )
        )

def _process_external_call(
    contract,
    function,
    external_call,
    contract_functions,
    external_calls,
    all_contracts,
    filename_input
):
    external_contract, external_function = external_call
    if not external_contract in all_contracts:
        return

    if isinstance(external_function, (Variable)):
        contract_functions[external_contract].add(tuple(
                _function_node(external_contract, external_function, filename_input).items()))

    external_calls.add(
        (
            tuple(_function_node(contract, function, filename_input).items()),
            tuple(_function_node(external_contract, external_function, filename_input).items()),
            'external_call',
            'external_call'
        )
    )

def _process_function(
    contract,
    function,
    contract_functions,
    contract_calls,
    solidity_functions,
    solidity_calls,
    external_calls,
    all_contracts,
    filename_input
):
    contract_functions[contract].add(tuple(
        _function_node(contract, function, filename_input).items())
    )
    for internal_call in function.internal_calls:
        _process_internal_call(
            contract,
            function,
            internal_call,
            contract_calls,
            solidity_functions,
            solidity_calls,
            filename_input
        )
    for external_call in function.high_level_calls:

        _process_external_call(
            contract,
            function,
            external_call,
            contract_functions,
            external_calls,
            all_contracts,
            filename_input
        )

def _process_functions(functions, filename_input, vulnerabilities_in_sc=None):
    contract_functions = defaultdict(set)  # contract -> contract functions nodes
    contract_calls = defaultdict(set)  # contract -> contract calls edges

    solidity_functions = set()  # solidity function nodes
    solidity_calls = set()  # solidity calls edges

    external_calls = set()  # external calls edges

    all_contracts = set()
    for function in functions:
        all_contracts.add(function.contract_declarer)

    for function in functions:
        _process_function(
            function.contract_declarer,
            function,
            contract_functions,
            contract_calls,
            solidity_functions,
            solidity_calls,
            external_calls,
            all_contracts,
            filename_input
        )

    all_contracts_graph = nx.MultiDiGraph()
    for contract in all_contracts:
        if len(contract_functions[contract]) > 0:
            for contract_function in contract_functions[contract]:
                node_id, node_label, node_type, function_fullname, contract_name, source_file, \
                node_function_source_code_start, node_function_source_code_length, source_visibility = _get_node_info(contract_function)

                all_contracts_graph.add_node(node_id, label=node_label, node_type=node_type,
                                  node_source_code_start=node_function_source_code_start, node_source_code_length=node_function_source_code_length,
                                  function_fullname=function_fullname, function_vis=source_visibility, contract_name=contract_name,
                                  source_file=source_file)

        if len(contract_calls[contract]) > 0:
            for contract_call in contract_calls[contract]:
                _add_edge_info_to_nxgraph(contract_call, all_contracts_graph)

    if len(external_calls) > 0:
        for external_call in external_calls:
            _add_edge_info_to_nxgraph(external_call, all_contracts_graph)

    return all_contracts_graph

## Generate control-flow graph

In [6]:
def get_node_info(node):
    node_label = "Node Type: {}\n".format(str(node.type))
    node_type = str(node.type)
    if node.expression:
        node_label += "\nEXPRESSION:\n{}\n".format(node.expression)
        node_expression = str(node.expression)
    else:
        node_expression = None
    if node.irs:
        node_label += "\nIRs:\n" + "\n".join([str(ir) for ir in node.irs])
        node_irs = "\n".join([str(ir) for ir in node.irs])
    else:
        node_irs = None

    # print(node_label)
    node_source_code_start = node.source_mapping['start']
    node_source_code_length = node.source_mapping['length']

    return node_label, node_type, node_expression, node_irs, node_source_code_start, node_source_code_length

## Merging Cfgs to Fcgs

In [7]:
def mapping_cfg_and_cg_node_labels(cfg, call_graph):
    dict_node_label_cfg_and_cg = {}

    for node, node_data in cfg.nodes(data=True):
        if node_data['node_type'] == 'FUNCTION_NAME':
            if node_data['label'] not in dict_node_label_cfg_and_cg:
                dict_node_label_cfg_and_cg[node_data['label']] = None

            dict_node_label_cfg_and_cg[node_data['label']] = {
                'cfg_node_id': node,
                'cfg_node_type': node_data['node_type']
            }

    for node, node_data in call_graph.nodes(data=True):
        if node_data['label'] in dict_node_label_cfg_and_cg:
            dict_node_label_cfg_and_cg[node_data['label']]['call_graph_node_id'] = node
            dict_node_label_cfg_and_cg[node_data['label']]['call_graph_node_type'] = node_data['node_type'].upper()
        else:
            print(node_data['label'], ' is not existing.')

    temp_dict = dict(dict_node_label_cfg_and_cg)
    for key, value in temp_dict.items():
        if 'call_graph_node_id' not in value or 'call_graph_node_type' not in value:
            dict_node_label_cfg_and_cg.pop(key, None)

    return dict_node_label_cfg_and_cg

def add_new_cfg_edges_from_call_graph(cfg, dict_node_label, call_graph):
    list_new_edges_cfg = []
    for source, target, edge_data in call_graph.edges(data=True):
        source_cfg = None
        target_cfg = None
        edge_data_cfg = edge_data
        for value in dict_node_label.values():
            if value['call_graph_node_id'] == source:
                source_cfg = value['cfg_node_id']

            if value['call_graph_node_id'] == target:
                target_cfg = value['cfg_node_id']

        if source_cfg is not None and target_cfg is not None:
            list_new_edges_cfg.append((source_cfg, target_cfg, edge_data_cfg))

    cfg.add_edges_from(list_new_edges_cfg)

    return cfg

def update_cfg_node_types_by_call_graph_node_types(cfg, dict_node_label):
    for value in dict_node_label.values():
        cfg_node_id = value['cfg_node_id']
        cfg.nodes[cfg_node_id]['node_type'] = value['call_graph_node_type']

## Call Graph Only

In [8]:
def get_call_graph(contract_path):
    sc_version = '0.4.25'
    pattern =  re.compile(r'\d.\d.\d+')
    with open(contract_path, 'r') as f:
        line = f.readline()
        while line:
            if 'pragma solidity' in line:
                if len(pattern.findall(line)) > 0:
                    sc_version = pattern.findall(line)[0]
                    break
                else:
                    sc_version = '0.4.25'
            line = f.readline()

    print(sc_version)
    solc_compiler = f'/content/ge-sc/artifacts/solc-{sc_version}'
    if not os.path.exists(solc_compiler):
        solc_compiler = f'/content/ge-sc/artifacts/solc-0.4.25'

    try:
        slither = Slither(contract_path, solc=solc_compiler)
    except Exception as e:
        print(e)
        return

    # Extract call graph
    all_functionss = [compilation_unit.functions for compilation_unit in slither.compilation_units]
    all_functions = [item for sublist in all_functionss for item in sublist]
    all_functions_as_dict = {function.canonical_name: function for function in all_functions}

    file_name_sc = contract_path.split('/')[-1:][0]
    all_contracts_call_graph = _process_functions(all_functions_as_dict.values(), file_name_sc)

    return all_contracts_call_graph

# Sol Files to Fcg Files

In [9]:
from transformers import RobertaTokenizer, RobertaModel
import torch



# Load the tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("Quangnguyen711/codebert-syntax-solidity-re-entrancy")
model = RobertaModel.from_pretrained("Quangnguyen711/codebert-syntax-solidity-re-entrancy")

def get_embeddings(text):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)

    # Get the model output
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the embeddings (we use the embeddings of the [CLS] token)
    embeddings = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

    return embeddings

def extract_function_code(source_file, start, length):
    with open(source_file, 'r') as f:
        source_code = f.read()

    # Extract the function's code
    function_code = source_code[start:start + length]

    return function_code

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/999k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [10]:
!mkdir /kaggle/working/SmartContractVulnerabilityDetection

In [11]:
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset")

os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg")

os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg")

In [12]:
def processSolFile(solFileSrc, fcgFileDst):
    try:
        tmp = Path(fcgFileDst) / f'{Path(solFileSrc).stem}.fcg'
        print(solFileSrc)
        G = nx.DiGraph(get_call_graph(solFileSrc))
        if len(G.nodes()) == 0:
            print("Compiler fail on:", solFileSrc)
        else:
            mappings, mappingsH = {}, {}
            katz = nx.katz_centrality(G)
            closeness = nx.closeness_centrality(G)
            clustering = nx.clustering(G)


            for node, data in G.nodes(data=True):
                mappings[node] = [G.in_degree(node),
                                              G.out_degree(node),
                                              katz[node],
                                              closeness[node],
                                              clustering[node]]

                code_start = data['node_source_code_start']
                code_length = data['node_source_code_length']
                mappingsH[node] =  get_embeddings(extract_function_code(solFileSrc, code_start, code_length))

            nx.set_node_attributes(G, mappings, 'features')
            nx.set_node_attributes(G, mappingsH, 'featuresH')

            cg = nx.convert_node_labels_to_integers(G)
            dg = dgl.from_networkx(cg, node_attrs=['features', 'featuresH'])
            fcgFileDst = Path(fcgFileDst) / f'{Path(solFileSrc).stem}.fcg'
            if os.path.exists(str(fcgFileDst)) == False:
                dgl.data.utils.save_graphs(str(fcgFileDst), [dg])
                print(f"Processed {Path(solFileSrc)}")

    except Exception as e:
        print('Error processing Sol file to FCG file:', str(e))


src = "/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
src2 = "/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
types = ["ReentrancyDataset"]
common_path = ["Test/NonVulnerable", "Test/Vulnerable", "Train/NonVulnerable", "Train/Vulnerable"]
save_common = ["Test/NonVulnerable_Fcg", "Test/Vulnerable_Fcg", "Train/NonVulnerable_Fcg", "Train/Vulnerable_Fcg"]
for tp in types:
    for id in range(len(common_path)):
        source = src + "/" + tp + "/" + common_path[id]
        destination = src2 + "/" + tp + "/" + save_common[id]

        print(source)
        print(destination)
        print("-" * 100)
        # Example usage
        solFileSrc = source
        solFiles = glob.glob(solFileSrc + "/*.sol")
        fcgFileDst = destination
        
        sol = [os.path.basename(file_link)[:-4] for file_link in solFiles]
        fcg = [file[:-4] for file in os.listdir(fcgFileDst)]
        
        check_point = set(sol) - set(fcg)
        
        cp_solFiles = []
        
        for file_link in solFiles:
            if os.path.basename(file_link).rstrip(".sol") in check_point:
                cp_solFiles.append(file_link)
        print(len(cp_solFiles))
        pool = ThreadPool(4)
        pool.map(partial(processSolFile, fcgFileDst=fcgFileDst), cp_solFiles)

/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable
/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg
----------------------------------------------------------------------------------------------------
74
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x6c9ac05c04a7a83f8afd4164f8e932dffdf69ffb.sol/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x33ea1d36d800e58dd87778ff4447854fc6b6d49c.sol/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0xa3db33ccfe990fdf89ff311754391b5c3af4ef04.sol


/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x33ea1d36d800e58dd87778ff4447854fc6b6d49c.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x11f4306f9812b80e75c1411c1cf296b04917b2f0.sol
0.4.24
Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x11f4306f9812b80e75c1411c1cf296b04917b2f0.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x8b7b6c61238088593bf75eec8fbf58d0a615d30c.sol
0.4.24


/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x6c9ac05c04a7a83f8afd4164f8e932dffdf69ffb.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x5fcc77ce412131daeb7654b3d18ee89b13d86cbf.sol
0.4.19


...

Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0x7da82c7ab4771ff031b66538d2fb9b0b047f6cf9.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0xec987914ade432ce9806f418787a4ed0b0e77000.sol
0.4.18


/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0x2c594e1cb006e86c3879b1d8191a8b059af52be7.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0x28b61faf5f4b9381a9cdb38d9f87788c563e3644.sol
0.4.25


/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0x40b10014a17e997e8e55594cbfb4f085c5ec815b.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0x9bbb97ae86075804c7802ba8b8b574b923dfe9b4.sol
0.4.24
Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0xec987914ade432ce9806f418787a4ed0b0e77000.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0xccf6450724e3271a62f1fa751a381fb3c58e68f2.sol
0.4.14
Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0xccf6450724e3271a62f1fa751a381fb3c58e68f2.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulne

In [13]:
# Load the tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("Quangnguyen711/codebert-syntax-solidity-time-dep")
model = RobertaModel.from_pretrained("Quangnguyen711/codebert-syntax-solidity-time-dep")

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/999k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [14]:
src = "/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
src2 = "/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
types = ["TimestampDependencyDataset"]
common_path = ["Test/NonVulnerable", "Test/Vulnerable", "Train/NonVulnerable", "Train/Vulnerable"]
save_common = ["Test/NonVulnerable_Fcg", "Test/Vulnerable_Fcg", "Train/NonVulnerable_Fcg", "Train/Vulnerable_Fcg"]
for tp in types:
    for id in range(len(common_path)):
        source = src + "/" + tp + "/" + common_path[id]
        destination = src2 + "/" + tp + "/" + save_common[id]

        print(source)
        print(destination)
        print("-" * 100)
        # Example usage
        solFileSrc = source
        solFiles = glob.glob(solFileSrc + "/*.sol")
        fcgFileDst = destination
        
        sol = [os.path.basename(file_link)[:-4] for file_link in solFiles]
        fcg = [file[:-4] for file in os.listdir(fcgFileDst)]
        
        check_point = set(sol) - set(fcg)
        
        cp_solFiles = []
        
        for file_link in solFiles:
            if os.path.basename(file_link).rstrip(".sol") in check_point:
                cp_solFiles.append(file_link)
        print(len(cp_solFiles))
        pool = ThreadPool(4)
        pool.map(partial(processSolFile, fcgFileDst=fcgFileDst), cp_solFiles)

/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable
/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg
----------------------------------------------------------------------------------------------------
73
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0xa06c318e59237f6ba20411ca7bccefd7e5357896.sol/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0xb0d926c1bc3d78064f3e1075d5bd9a24f35ae6c5.sol/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0xb3775fb83f7d12a36e0475abdd1fca35c091efbe.sol/kaggle/input/sc-vuldetection-dataset/SmartContractVulne

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0xa06c318e59237f6ba20411ca7bccefd7e5357896.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0x2d3b62a8b756103e417d161c20e97e76ed5ef0c2.sol
0.4.15
Invalid compilation: 
Invalid solc compilation /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0x2d3b62a8b756103e417d161c20e97e76ed5ef0c2.sol:197:44: Error: Expected primary expression.
 bytes32 myQueryId = oraclize_query("URL", "json(https:
                                           ^

Compiler fail on: /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0x2d3b62a8b756103e417d161c20e97e76ed5ef0c2.sol
/kaggle/

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0xb0d926c1bc3d78064f3e1075d5bd9a24f35ae6c5.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0xa25e8050f80ee99a17e861cd0931d5d362caa34e.sol
0.4.18
Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0x885a4819e899c772b439f05944096a3236315550.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0xe64287516518eda9f7092a0626cba00baf21a301.sol
0.4.13
Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0xe64287516518eda9f7092a0626cba00baf21a301.sol
/

...

Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x2183481bf5fda35ca45e85d380603def3bc069e1.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x07Ec6c3159c2336Ba36Ab41f73411f8fEe430470.sol
0.4.15
Invalid compilation: 
Invalid solc compilation /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x07Ec6c3159c2336Ba36Ab41f73411f8fEe430470.sol:2:2: Error: Expected pragma, import directive or contract/interface/library definition.
 function Ownable() {
 ^

Compiler fail on: /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x07Ec6c3159c2336Ba36Ab41f73411f8fEe430470.sol
/kaggle/input/sc-vuldetection-dataset/SmartContr

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0xf12a2e2a1a1d714d6c7db114806411596a09b10a.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x7bd52bdff0acf4e18dd80c6ee86db205d83f88ce.sol
0.4.18
Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x7bd52bdff0acf4e18dd80c6ee86db205d83f88ce.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x322909bb3aa921f101d829c0edf57493468d9bd4.sol
0.4.18
Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x322909bb3aa921f101d829c0edf57493468d9bd4.sol
/kaggle/inp

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x063425e215701d2761a9065e647fa98f209b4ddd.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x8fa1EaD5d8d774b27d288711abE4d4258224ae26.sol
0.4.11
Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x3744942C42451c2B42F43a51eE9bB6c6ad0FDc86.sol
/kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x5bc7e5f0ab8b2e10d2d0a3f21739fce62459aef3.sol
0.4.17
Processed /kaggle/input/sc-vuldetection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x2AbF00B596f4cB10384654DCD76253B3140d35Ff.sol
/kaggle/inp